In [ ]:
import gmsh

gmsh.initialize()
gmsh.model.add("L_profile_3D")

# ----------------------------------------------------------------------
# Parameters
# ----------------------------------------------------------------------

# vertical leg thickness=17.5, horizontal leg length = 17.5 + 17, c=17 and d=23/2.
# a=17.5, b=17, c=17 and d=23/2.

# Values as shown on schematic
a=17.5
b=17
c=17
d=23/2

horizontalLegLength = a+b
verticalLegLength = c+d

verticalLegThickness = a
horizontalLegThickness = d

depth = 5 # extrusion depth

assert verticalLegThickness < horizontalLegLength
assert horizontalLegThickness < verticalLegLength

lc = min(verticalLegThickness, horizontalLegThickness) / 3

# ----------------------------------------------------------------------
# 2D profile
# ----------------------------------------------------------------------

p1 = gmsh.model.geo.addPoint(0,   0,   0, lc)
p2 = gmsh.model.geo.addPoint(horizontalLegLength,   0,   0, lc)
p3 = gmsh.model.geo.addPoint(horizontalLegLength,  horizontalLegThickness,   0, lc)
p4 = gmsh.model.geo.addPoint(verticalLegThickness, horizontalLegThickness,   0, lc)
p5 = gmsh.model.geo.addPoint(verticalLegThickness,  verticalLegLength,   0, lc)
p6 = gmsh.model.geo.addPoint(0,   verticalLegLength,   0, lc)

l1 = gmsh.model.geo.addLine(p1, p2)
l2 = gmsh.model.geo.addLine(p2, p3)
l3 = gmsh.model.geo.addLine(p3, p4)
l4 = gmsh.model.geo.addLine(p4, p5)
l5 = gmsh.model.geo.addLine(p5, p6)
l6 = gmsh.model.geo.addLine(p6, p1)

loop = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4, l5, l6])
surf = gmsh.model.geo.addPlaneSurface([loop])

gmsh.model.geo.synchronize()

# ----------------------------------------------------------------------
# Extrude
# ----------------------------------------------------------------------

out = gmsh.model.geo.extrude([(2, surf)], 0, 0, depth)

gmsh.model.geo.synchronize()

# ----------------------------------------------------------------------
# Identify entities
# ----------------------------------------------------------------------

left_faces = []
bottom_faces = []

eps = 1e-8

for dim, tag in gmsh.model.getEntities(2):

    xmin, ymin, zmin, xmax, ymax, zmax = gmsh.model.getBoundingBox(dim, tag)

    # Entire left surface (x = 0)
    if abs(xmin) < eps and abs(xmax) < eps:
        left_faces.append(tag)

    # Entire bottom surface (y = 0)
    if abs(ymin) < eps and abs(ymax) < eps:
        bottom_faces.append(tag)

volumes = [tag for _, tag in gmsh.model.getEntities(3)]

# ----------------------------------------------------------------------
# Physical groups
# ----------------------------------------------------------------------

gmsh.model.addPhysicalGroup(2, left_faces, 1)
gmsh.model.setPhysicalName(2, 1, "left")

gmsh.model.addPhysicalGroup(2, bottom_faces, 2)
gmsh.model.setPhysicalName(2, 2, "bottom")

gmsh.model.addPhysicalGroup(3, volumes, 10)
gmsh.model.setPhysicalName(3, 10, "volume")

# ----------------------------------------------------------------------
# Mesh
# ----------------------------------------------------------------------

gmsh.option.setNumber("Mesh.CharacteristicLengthMin", lc)
gmsh.option.setNumber("Mesh.CharacteristicLengthMax", lc)

gmsh.model.mesh.generate(3)

gmsh.write("L_profile_3D.msh")

# gmsh.fltk.run()

# gmsh.finalize()


In [ ]:
from dolfinx.io import gmsh as gmshio
from dolfinx.fem.petsc import LinearProblem
from mpi4py import MPI

gdim = 3
gmsh_model_rank = 0
mesh_comm = MPI.COMM_WORLD
mesh_data = gmshio.model_to_mesh(gmsh.model, mesh_comm, gmsh_model_rank, gdim=gdim)
assert mesh_data.cell_tags is not None
cell_markers = mesh_data.cell_tags
domain = mesh_data.mesh

In [ ]:
import pyvista
import dolfinx
topology, cell_types, points = dolfinx.plot.vtk_mesh(domain)
pv_mesh = pyvista.UnstructuredGrid(topology, cell_types, points)
pv_mesh.plot(show_edges=True)